# Notebook 01: Carga y unificacion del dataset multi-planta

## Descripcion
Este notebook unifica los CSV procesados de las tres plantas fotovoltaicas
(Afrisol, E03 y LECA1) en un unico DataFrame validado, realiza un analisis
exploratorio inicial y guarda el resultado como punto de entrada para todos
los notebooks de entrenamiento y evaluacion.

## Flujo del notebook
1. Localizar la raiz del proyecto y configurar rutas
2. Importar funciones de `src`
3. Cargar los CSV de cada planta y unificar
4. Resumen estadistico y consistencia temporal
5. Analisis exploratorio: distribuciones y perfil horario
6. Deteccion y eliminacion de anomalias (radiation = 999.0)
7. Verificacion de nulos y negativos
8. Analisis estacional
9. Guardar el CSV unificado

**Este notebook solo prepara y valida el dataset unificado. No entrena modelos.**

## Puntos clave
- Resolucion temporal de 15 minutos, periodo enero 2022 - mayo 2024.
- Se eliminan registros con `radiation = 999.0` (valor erroneo del sensor).
- El CSV de salida (`data/df_all_unificado.csv`) es el punto de entrada de todos los notebooks siguientes.


### 1. Configuracion de rutas
Se localiza la raiz del repositorio de forma automatica buscando las carpetas
`data/` y `src/` hacia arriba desde el directorio actual.


In [1]:
import os
import sys
from pathlib import Path

current = Path().resolve()
root = current.anchor

while not ((current / "data").exists() and (current / "src").exists()):
    if str(current) == root:
        raise FileNotFoundError(
            "No se encontro la raiz del proyecto. "
            "Asegurate de que existen las carpetas data/ y src/."
        )
    current = current.parent

os.chdir(current)
if str(current) not in sys.path:
    sys.path.insert(0, str(current))

PROJECT_ROOT = current
DATA_DIR = PROJECT_ROOT / "data"
print(f"Raiz del proyecto: {PROJECT_ROOT.name}")


Raiz del proyecto: TFM_Repositorio


### 2. Importacion de librerias y funciones


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

from src.data import (
    PlantDataset,
    basic_cleaning,
    check_time_consistency,
    load_multiple_plants,
    summarize_plants,
)

warnings.filterwarnings("ignore", category=FutureWarning)


### 3. Carga y unificacion de plantas
Las rutas son relativas a la raiz del repositorio, independientemente
del sistema operativo o entorno donde se ejecute el notebook.


In [ ]:
datasets = [
    PlantDataset("E03",     DATA_DIR / "Datos_E03_Con_Temperatura_OpenMeteo.csv"),
    PlantDataset("Afrisol", DATA_DIR / "Datos_Afrisol_Con_Temperatura_OpenMeteo_2.csv"),
    PlantDataset("LECA1",   DATA_DIR / "Datos_LECA1_Con_Temperatura_OpenMeteo.csv"),
]

df_all = load_multiple_plants(datasets)
df_all = basic_cleaning(df_all)

print(f"Shape inicial: {df_all.shape}")
df_all.head()


### 4. Resumen estadistico y consistencia temporal
`summarize_plants` ofrece una vision rapida del numero de registros, rango
temporal y valores nulos por planta. 

`check_time_consistency` cuantifica
los huecos respecto a la frecuencia esperada de 15 minutos.


In [ ]:
summary = summarize_plants(df_all)
display(summary)

time_check = check_time_consistency(df_all)
display(time_check)


### 5. Deteccion detallada de huecos temporales
Se identifican los huecos superiores a 15 minutos entre registros
consecutivos de cada planta.


In [ ]:
def find_time_gaps(df: pd.DataFrame, freq: str = "15min") -> pd.DataFrame:
    """Devuelve los huecos temporales superiores a freq por planta."""
    df = df.sort_values(["id_planta", "timestamp"]).copy()
    expected = pd.to_timedelta(freq)
    gaps = []
    for plant_id, g in df.groupby("id_planta"):
        diffs = g["timestamp"].diff()
        mask = diffs != expected
        mask.iloc[0] = False
        g_gaps = g.loc[mask].copy()
        g_gaps["prev_timestamp"] = g["timestamp"].shift(1)[mask]
        g_gaps["gap_minutes"] = diffs[mask].dt.total_seconds() / 60
        g_gaps["plant"] = plant_id
        gaps.append(g_gaps[["plant", "prev_timestamp", "timestamp", "gap_minutes"]])
    return pd.concat(gaps, ignore_index=True) if gaps else pd.DataFrame()

gaps = find_time_gaps(df_all)
print(f"Total de huecos encontrados: {len(gaps)}")
display(gaps)


### 6. Analisis exploratorio: distribuciones y perfil horario
Distribuciones de irradiancia y potencia por planta y perfil horario
medio de produccion.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(data=df_all, x="radiation", hue="id_planta", bins=50, ax=axes[0])
axes[0].set_title("Distribucion de irradiancia por planta")

sns.histplot(data=df_all, x="power", hue="id_planta", bins=50, ax=axes[1])
axes[1].set_title("Distribucion de potencia por planta")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
df_all.groupby(["id_planta", df_all["timestamp"].dt.hour])["power"].mean() \
    .unstack().T.plot(ax=ax)
ax.set_title("Perfil horario medio de potencia por planta")
ax.set_xlabel("Hora del dia")
ax.set_ylabel("Potencia media (W)")
plt.tight_layout()
plt.show()


### 7. Deteccion y eliminacion de anomalias: radiation = 999.0
Se detectaron registros con `radiation = 999.0`, codigo de error del sensor.
Se concentran en el dia del ano 125 (principios de mayo).
Se eliminan todos los registros con este valor antes de continuar.


In [ ]:
anomalous = df_all[df_all["radiation"] == 999.0]
print(f"Registros con radiation=999: {len(anomalous)}")
print("Distribucion por day_of_year:")
display(anomalous["day_of_year"].value_counts().head(10))

df_all_clean = df_all[df_all["radiation"] != 999.0].copy()
print(f"Shape tras limpieza: {df_all_clean.shape} "
      f"(eliminados {len(df_all) - len(df_all_clean)} registros)")


### 8. Verificacion de nulos y negativos


In [ ]:
print("Nulos por columna:")
display(df_all_clean.isna().sum())

print(f"\nNegativos en radiation: {(df_all_clean['radiation'] < 0).sum()}")
print(f"Negativos en power:     {(df_all_clean['power'] < 0).sum()}")

print("\nResumen estadistico:")
display(df_all_clean[["radiation", "power", "T_ambiente"]].describe())


### 9. Analisis estacional
Produccion media por estacion y planta, restringida a horas de luz
(registros con `power > 0`).


In [ ]:
season_map = {
    12: "Invierno", 1: "Invierno",  2: "Invierno",
     3: "Primavera", 4: "Primavera", 5: "Primavera",
     6: "Verano",    7: "Verano",    8: "Verano",
     9: "Otono",    10: "Otono",    11: "Otono",
}
season_order = ["Invierno", "Primavera", "Verano", "Otono"]

df_all_clean = df_all_clean.copy()
df_all_clean["season"] = df_all_clean["timestamp"].dt.month.map(season_map)

df_season_agg = (
    df_all_clean[df_all_clean["power"] > 0]
    .groupby(["id_planta", "season"])["power"]
    .mean().reset_index()
    .rename(columns={"power": "mean_power"})
)
df_season_agg["season"] = pd.Categorical(
    df_season_agg["season"], categories=season_order, ordered=True
)

fig = px.bar(
    df_season_agg.sort_values("season"),
    x="season", y="mean_power", color="id_planta", barmode="group",
    title="Produccion media por estacion y planta (horas de luz)",
    labels={"mean_power": "Potencia media (W)", "season": "Estacion"},
)
fig.show()


### 10. Guardar el dataset unificado
Se elimina la columna auxiliar `season` antes de exportar.
El archivo generado es la entrada de todos los notebooks de entrenamiento.


In [ ]:
output_path = DATA_DIR / "df_all_unificado.csv"

df_export = df_all_clean.drop(columns=["season"], errors="ignore")
df_export.to_csv(output_path, index=False)

print(f"Archivo guardado en: data/{output_path.name}")
print(f"Filas: {len(df_export):,} | Columnas: {df_export.shape[1]}")
print(f"Plantas: {sorted(df_export['id_planta'].unique())}")
print(f"Rango temporal: {df_export['timestamp'].min()} -> {df_export['timestamp'].max()}")


In [ ]:
df_export.info()


## Conclusiones del notebook 01

| Aspecto | Resultado |
|---|---|
| Plantas | Afrisol, E03, LECA1 |
| Registros finales | ~249.285 filas |
| Periodo cubierto | Enero 2022 - Mayo 2024 |
| Resolucion temporal | 15 minutos |
| Anomalias eliminadas | radiation=999.0 (principalmente dia 125) |
| Huecos temporales | Detectados y documentados (2 por planta) |

**Archivo de salida:** `data/df_all_unificado.csv`  
**Punto de entrada del siguiente notebook:** construccion de features y entrenamiento de modelos.
